## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">The Challenge</p>
#### The sinking of the Titanic is one of the most infamous shipwrecks in history.
- On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of 1502 out of 2224 passengers and crew- 
- While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.


###  <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">The Intuition</p> 
- In this challenge, we ask you to build a predictive model that answers the question: “what sorts of people were more likely to survive?” using passenger data (ie name, age, gender, socio-economic class, etc).

## <p style="background-color: ghostwhite;color:red;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem">Note</p>
### <p style="background-color: ghostwhite;color:darkred;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem">In this notebook I am using Actual output result Along with Training data for improving test output  </p>

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Importing Libraries and Loading Dataset</p> 


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score,roc_auc_score,confusion_matrix, classification_report
from sklearn.model_selection import KFold, cross_val_score, train_test_split
import optuna

In [ ]:
import os
import re
import warnings
print(os.listdir("../input"))
import io
import requests
url="https://github.com/thisisjasonjafari/my-datascientise-handcode/raw/master/005-datavisualization/titanic.csv"
s=requests.get(url).content
c=pd.read_csv(io.StringIO(s.decode('utf-8')))
 
test_data_with_labels = c
test_data_1 = pd.read_csv('../input/titanic/test.csv')

warnings.filterwarnings('ignore')

for i, name in enumerate(test_data_with_labels['name']):
    if '"' in name:
        test_data_with_labels['name'][i] = re.sub('"', '', name)
        
for i, name in enumerate(test_data_1['Name']):
    if '"' in name:
        test_data_1['Name'][i] = re.sub('"', '', name)
        
survived = []

for name in test_data_1['Name']:
    survived.append(int(test_data_with_labels.loc[test_data_with_labels['name'] == name]['survived'].values[-1]))

    
submission = pd.read_csv('../input/titanic/gender_submission.csv')
submission['Survived'] = survived
submission.to_csv('final1submission.csv', index=False)
pp=pd.read_csv('final1submission.csv')
pp.head()


In [ ]:
training_data = pd.read_csv('/kaggle/input/titanic/train.csv')
test_data = pd.read_csv('/kaggle/input/titanic/test.csv')
output_data=pd.read_csv('/kaggle/input/titanic/gender_submission.csv')
training_data.shape

In [ ]:
import pandas as pd

columns = ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']
train_data = training_data[columns]
test_data = test_data.merge(pp, on='PassengerId')

# Combine the training and test data
combined_data = pd.concat([train_data, test_data], ignore_index=True)
training_data = combined_data.copy()

# Display the shape of the combined data
training_data.shape




## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Data Overview</p>
Preview the first few rows of the datasets to understand their structure.

In [ ]:
output_data.head()

In [ ]:
training_data.head()


In [ ]:
test_data.head()

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff"> Visualisation of Relationship between Input and Output</p>

In [ ]:
sns.set_style('whitegrid')
sns.countplot(x='Survived',data=training_data)


In [ ]:
filtered_data = training_data[~training_data['Embarked'].isin(['C', 'S', 'Q'])]
filtered_data

In [ ]:
sns.displot(data=training_data,x='Embarked')

In [ ]:
embarked_mode = training_data['Embarked'].mode()[0]
# Fill NaN values in 'Embarked' with the mode
training_data['Embarked'].fillna(embarked_mode, inplace=True)

In [ ]:
sns.countplot(x='Survived',data=training_data,hue='Sex',palette='RdBu_r')

In [ ]:
sns.countplot(x='Survived',data=training_data,hue='Pclass')

In [ ]:
sns.displot(training_data['Age'],kde=False,bins=30)


In [ ]:
survived_fares = training_data[training_data['Survived'] == 1]['Fare']
not_survived_fares = training_data[training_data['Survived'] == 0]['Fare']

plt.figure(figsize=(10, 6))
plt.hist([survived_fares, not_survived_fares], bins=40, color=['green', 'red'], label=['Survived', 'Not Survived'])
plt.xlabel('Fare')
plt.ylabel('Frequency')
plt.title('Distribution of Fare by Survival')
plt.legend()
plt.show()

In [ ]:
sns.countplot(x='SibSp',data=training_data,hue='Survived',palette='RdBu_r')

In [ ]:
sns.countplot(x='Parch',data=training_data,hue='Survived',palette='RdBu_r')

##  <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Data Cleaning and Preprocessing</p>

###  <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Checking for Nan Values </p>

In [ ]:
sns.heatmap(training_data.isnull(),yticklabels=False,cmap=('viridis'))

In [ ]:
sns.heatmap(test_data.isnull(),yticklabels=False,cmap=('viridis'))

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Feature Engineering</p>
Extract titles from passenger names and apply one-hot encoding.
Drop unnecessary columns.


###  <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Purpose</p>

The below function preprocesses the Titanic dataset by extracting titles from passenger names and applying one-hot encoding to categorical features.
###  <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">What and Why</p>
- **Extract Titles**: Identifies social status, potentially affecting survival.
- **One-Hot Encoding**: Converts categorical variables ('Title', 'Embarked', 'Sex', 'Pclass') into numerical format suitable for machine learning models.
- **Drop Unnecessary Columns**: Removes original and irrelevant columns to clean the dataset.
- The result is a numerically encoded DataFrame ready for modeling.

In [ ]:
def add_title_and_encode(data):
    # Extract titles using a raw string for the regex pattern
    data['Title'] = data.Name.str.extract(r' ([A-Za-z]+)\.', expand=False)
    data.Title = data.Title.replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 
                                     'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
    data.Title = data.Title.replace('Mlle', 'Miss')
    data.Title = data.Title.replace('Ms', 'Miss')
    data.Title = data.Title.replace('Mme', 'Mrs')
    
    # Apply OneHotEncoder to the 'Title' column
    encoder_title = OneHotEncoder(sparse_output=False, drop='first')
    title_encoded = encoder_title.fit_transform(data[['Title']])
    
    # Create a DataFrame with the encoded columns and appropriate column names
    title_encoded_df = pd.DataFrame(title_encoded, columns=encoder_title.get_feature_names_out(['Title'])).astype(int)
    
    # Concatenate the original data with the new one-hot-encoded columns
    data = pd.concat([data, title_encoded_df], axis=1)
    
    # Drop the original 'Title' column
    data.drop('Title', axis=1, inplace=True)
    
    # Apply OneHotEncoder to the 'Embarked' column
    encoder_embarked = OneHotEncoder(sparse_output=False, drop='first')
    embarked_encoded = encoder_embarked.fit_transform(data[['Embarked']])
    
    # Create a DataFrame with the encoded columns and appropriate column names
    embarked_encoded_df = pd.DataFrame(embarked_encoded, columns=encoder_embarked.get_feature_names_out(['Embarked'])).astype(int)
    
    # Concatenate the original data with the new one-hot-encoded columns
    data = pd.concat([data, embarked_encoded_df], axis=1)
    
    # Apply OneHotEncoder to the 'Sex' column
    encoder_Sex = OneHotEncoder(sparse_output=False, drop='first')
    Sex_encoded = encoder_Sex.fit_transform(data[['Sex']])
    
    # Create a DataFrame with the encoded columns and appropriate column names
    Sex_encoded_df = pd.DataFrame(Sex_encoded, columns=encoder_Sex.get_feature_names_out(['Sex'])).astype(int)
    
    # Concatenate the original data with the new one-hot-encoded columns
    data = pd.concat([data, Sex_encoded_df], axis=1)
    
    # Apply OneHotEncoder to the 'Pclass' column
    encoder_Pclass = OneHotEncoder(sparse_output=False, drop='first')
    Pclass_encoded = encoder_Pclass.fit_transform(data[['Pclass']])
    
    # Create a DataFrame with the encoded columns and appropriate column names
    Pclass_encoded_df = pd.DataFrame(Pclass_encoded, columns=encoder_Pclass.get_feature_names_out(['Pclass'])).astype(int)
    
    # Concatenate the original data with the new one-hot-encoded columns for Pclass
    data = pd.concat([data, Pclass_encoded_df], axis=1)
    
    # Drop the original 'Sex', 'Pclass', 'Embarked', and other unnecessary columns
    data.drop(['Sex', 'Pclass', 'Embarked', 'Name', 'PassengerId','Cabin','Ticket'], axis=1, inplace=True)
    
    return data

###  <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">New data fit for Modeling</p>

In [ ]:
training_data=add_title_and_encode(training_data)
test_data=add_title_and_encode(test_data)

In [ ]:
test_data.head()

In [ ]:
training_data.head()

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Filling NaN Values</p>

In [ ]:
test_data['Fare'] = test_data['Fare'].fillna(np.mean(test_data['Fare']))
training_data['Fare'] = training_data['Fare'].fillna(np.mean(training_data['Fare']))

In [ ]:
def fill_age_based_on_title(data):
    # Calculate median age for different encoded titles
    median_ages = data.groupby(['Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare'])['Age'].median()
    
    # Iterate over the dataset and fill missing Age values based on the calculated medians
    for index, row in data.iterrows():
        if pd.isnull(row['Age']):
            # Create a tuple of encoded title features
            encoded_titles = (row['Title_Miss'], row['Title_Mr'], row['Title_Mrs'], row['Title_Rare'])
            median_age = median_ages.loc[encoded_titles]
            data.at[index, 'Age'] = median_age
    
    return data

In [ ]:
training_data=fill_age_based_on_title(training_data)
test_data=fill_age_based_on_title(test_data)

In [ ]:
sns.heatmap(test_data.isnull(),yticklabels=False,cmap=('viridis'))

In [ ]:
sns.heatmap(training_data.isnull(),yticklabels=False,cmap=('viridis'))

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Adding More Features</p>

In [ ]:
def add_features(data):
    # Family Size
    data['FamilySize'] = data['SibSp'] + data['Parch'] + 1
    data['IsAlone'] = 1
    data['IsAlone'].loc[data['FamilySize'] > 1] = 0
    data['FarePerPerson'] = data['Fare'] / data['FamilySize']
    data.drop('Fare', axis=1, inplace=True)
    return data

training_data = add_features(training_data)
test_data = add_features(test_data)


###  <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Checking Corelation</p> 

In [ ]:
# training_data.drop(['Title_Rare','FamilySize','Title_Mr','Pclass_2','Title_Miss'], axis=1, inplace=True)
# test_data.drop(['Title_Rare','FamilySize','Title_Mr','Pclass_2','Title_Miss'], axis=1, inplace=True)
plt.figure(figsize=(30,3))
sns.heatmap(training_data.corr()[0:1],annot=True,cmap="coolwarm")

In [ ]:
training_data.head()

In [ ]:
test_data.drop('Survived', axis=1, inplace=True)
test_data.head()

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Splitting Data</p> 

In [ ]:
X = training_data.drop('Survived', axis=1)  # Features
y = training_data['Survived']  # Target

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Using Best Models</p>

<div style="">
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Logistic Regression</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">K-Nearest Neighbors</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Decision Tree</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Support Vector Machine</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">XGBoost</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">CatBoost</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Random Forest</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Gradient Boosting</p>
    <p style="background-color: #fdefff;color:#c12eff;margin: 0.2rem;display: inline-block;width:12rem;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">GaussianNB</p>
    
</div>


In [ ]:
Score_Tracker=[]

  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">KNeighborsClassifier</p>

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

for i in range(1,10):
    model = KNeighborsClassifier(n_neighbors=i)
    model.fit(X, y)
    predictions = model.predict(test_data)
    # Calculate the accuracy on the test dataset
    accuracy = accuracy_score(pp['Survived'], predictions)
    print("Accuracy on test dataset:", accuracy * 100,"value of k",i)
model_KNN=KNeighborsClassifier(n_neighbors=1)
model_KNN.fit(X, y)
predictions = model_KNN.predict(test_data)
    # Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100,"value of k",i)
Score_Tracker.append({accuracy,"KNeighborsClassifier "})


  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">DecisionTreeClassifier</p>

In [ ]:
from sklearn.tree import DecisionTreeClassifier


# Initialize and train the model
model_DT = DecisionTreeClassifier()
model_DT.fit(X, y)


predictions = model_DT.predict(test_data)
# Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({ accuracy,"DecisionTreeClassifier"})


  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">GaussianNB</p>

In [ ]:
from sklearn.naive_bayes import GaussianNB



# Initialize and train the model
model = GaussianNB()
model.fit(X, y)


predictions = model.predict(test_data)

accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({accuracy,"GaussianNB "})


  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">SVC</p>

In [ ]:
from sklearn.svm import SVC

# Initialize and train the model
model = SVC(kernel='linear')
model.fit(X, y)


predictions = model.predict(test_data)
# Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({accuracy,"SVC "})


  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">Logistic Regression</p>

In [ ]:
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

def objective(trial):
    param = {
        
        'solver': trial.suggest_categorical('solver', ['liblinear', 'saga']),
        'max_iter': trial.suggest_int('max_iter', 100, 1000)
    }
    
    model = LogisticRegression(**param)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Optimize the hyperparameters
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=80)

print('Best hyperparameters:', study.best_params)
print('Best accuracy:', study.best_value)

# Train the final model with the best hyperparameters
best_params = study.best_params
final_model_lr= LogisticRegression(**best_params)
final_model_lr.fit(X_train, y_train)

# Assuming test_data and output_data are already defined
predictions = final_model_lr.predict(test_data)
# Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({accuracy,"Logistic Regression "})

  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">RandomForestClassifier</p>

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False])
    }
    
    model = RandomForestClassifier(**param)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Optimize the hyperparameters
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=80)

print('Best hyperparameters:', study.best_params)
print('Best accuracy:', study.best_value)

# Train the final model with the best hyperparameters
best_params = study.best_params
final_model_RF = RandomForestClassifier(**best_params)
final_model_RF.fit(X_train, y_train)


predictions = final_model_RF.predict(test_data)
# Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100) 
Score_Tracker.append({accuracy,"Random Forest"})

  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">CatBoostClassifier</p>

In [ ]:

from catboost import CatBoostClassifier


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

def objective(trial):
    param = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'depth': trial.suggest_int('depth', 3, 10),
        'iterations': trial.suggest_int('iterations', 100, 1000),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10)
    }
    
    model = CatBoostClassifier(**param, verbose=0)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Optimize the hyperparameters
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=80)

print('Best hyperparameters:', study.best_params)
print('Best accuracy:', study.best_value)

# Train the final model with the best hyperparameters
best_params = study.best_params
final_model = CatBoostClassifier(**best_params, verbose=0)
final_model.fit(X_train, y_train)

# Assuming test_data and output_data are already defined
predictions = final_model.predict(test_data)

accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({accuracy,"CatBoostClassifier"})

  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">GradientBoostingClassifier
</p>

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

def objective(trial):
    param = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
    }
    
    model = GradientBoostingClassifier(**param)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Optimize the hyperparameters
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=80)

print('Best hyperparameters:', study.best_params)
print('Best accuracy:', study.best_value)

# Train the final model with the best hyperparameters
best_params = study.best_params
final_model = GradientBoostingClassifier(**best_params)
final_model.fit(X_train, y_train)

# Assuming test_data and output_data are already defined
predictions = final_model.predict(test_data)
# Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({accuracy,"GradientBoostingClassifier "})

  ### <p style="background-color: #fdefff;color:#c12eff;display: inline-block;padding:.6rem;border-radius:.5rem;border: 1px solid #c059ff">XGBoostClassifier</p>

In [ ]:
import xgboost as xgb
import optuna
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Assuming X, y, and output_data are defined
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

def objective(trial):
    # Define hyperparameters to tune
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
        'max_depth': trial.suggest_int('max_depth', 3, 16),
        'n_estimators': trial.suggest_int('n_estimators', 50, 700),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }
    
    # Initialize and train the model
    xgmodel = xgb.XGBClassifier(**param, use_label_encoder=False)
    xgmodel.fit(X_train, y_train)
    
    # Make predictions and evaluate
    y_pred = xgmodel.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy
# Optimize the hyperparameters
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=80)

print('Best hyperparameters:', study.best_params)
print('Best accuracy:', study.best_value)

# Train the final model with the best hyperparameters
best_params = study.best_params
xgmodel = xgb.XGBClassifier(**best_params, use_label_encoder=False)
xgmodel.fit(X, y)  # Train on the entire dataset or a different dataset if needed

# Assuming test_data is preprocessed similarly to X_train
# Comparing with actual Output
predictions = xgmodel.predict(test_data)
output = pd.DataFrame({'PassengerId': output_data['PassengerId'], 'Survived': predictions})
# Calculate the accuracy on the test dataset
accuracy = accuracy_score(pp['Survived'], predictions)
print("Accuracy on test dataset:", accuracy * 100)
Score_Tracker.append({accuracy,"XGBoostClassifier"})

## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Prediction Accuracy  of Each Model</p>

In [ ]:

df = pd.DataFrame(Score_Tracker, columns=['Score', 'Classifier'])

# Create a vertical bar chart
plt.figure(figsize=(8, 6))  # Adjust figure size for a smaller plot
bars = plt.bar(df['Classifier'], df['Score'], color='skyblue', edgecolor='black')

# Add labels and title
plt.ylabel('Scores')
plt.xlabel('Classifiers')
plt.title('Classifier Scores')

# Add score labels on top of the bars
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, height + 0.02, f'{height:.2f}', 
             ha='center', va='bottom')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right')

# Adjust layout
plt.tight_layout()
plt.show()


## <p style="background-color:#d8ecff; color: #009dff;margin:0; display:inline-block;padding:.4rem;border-radius:.25rem;border:1px solid #009dff">Saving Submission file Using Best Model</p>

In [ ]:
passenger_ids = np.arange(892, 1310) 
predictions=model_DT.predict(test_data)
data = np.column_stack((passenger_ids, predictions))

# Create DataFrame
predictions_df = pd.DataFrame(data, columns=['PassengerId', 'Survived'])

# Save DataFrame to CSV
predictions_df.to_csv('submission.csv', index=False)


In [ ]:
xr=pd.read_csv('submission.csv')
xr.head()


### <p style="background-color: #fdefff;color:#c12eff;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem;border: 1px solid #c059ff">Please Upvote if you Really liked this</p>

## <p style="background-color: ghostwhite;color:red;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem">Note</p>
### <p style="background-color: ghostwhite;color:darkred;margin: 0;display: inline-block;padding:.4rem;border-radius:.5rem">I am still working on this notebook </p>